# Your first Qwen fine tuning experiment
**Google Colab + Unsloth + 0.6B Qwen3**

This is a small educational lab designed for someone with a Windows laptop with limited GPU RAM. The GPU work happens in Colab, not your laptop. Start by opening **Runtime > Change runtime type > GPU**. Run cells from top to bottom. The final GGUF export cell is optional.

**Important:** Colab GPU availability and Python package APIs change over time. The install/training cells may need to be aligned with Unsloth's latest maintained Qwen3 Colab example. This notebook was authored for learning and cannot be executed on this system's Colab GPU in advance.


## 01 | Check the cloud GPU
**Why:** Make sure Colab has allocated a GPU before you download anything. Your RTX 3050 at home is not being used by a cloud Colab runtime.

**What the code is doing:** `!nvidia-smi` asks the NVIDIA driver about the GPU and its memory. The exclamation mark tells the notebook to execute a Linux shell command. `import torch` loads PyTorch, the library performing the GPU calculations. `torch.cuda.is_available()` tests whether it can access an NVIDIA GPU. The `if` block prints the hardware name and memory only if the check passes. `1024**3` converts bytes into gibibytes.

**Expected:** Expect a GPU such as Tesla T4 or another NVIDIA model, and CUDA available: True. Exact GPU type and memory vary.

**Check:** If CUDA is False, stop. Choose Runtime > Change runtime type > GPU, reconnect, and rerun.

**If it fails:** If the GPU option is unavailable, free Colab quota or capacity may be exhausted. Try later; do not pay before testing the free tier.


In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU model:", torch.cuda.get_device_name(0))
    print("GPU VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


## 02 | Install software inside Colab
**Why:** Install the training libraries on the cloud computer. Installing them on Windows is not required.

**What the code is doing:** `%pip` installs Python packages into the notebook environment. `unsloth` loads and fine tunes models using memory-saving methods. `datasets` handles training records. `trl` provides a supervised fine tuning trainer. `-q` asks for quieter output; `-U` asks for current compatible releases. Package versions may change; this is the most version-sensitive step.

**Expected:** Installation finishes without a red error. You may see informational warnings about package versions.

**Check:** If asked to restart, use Runtime > Restart session, then start running from Cell 01 again.

**If it fails:** If dependencies conflict, use the installation cell from the latest official Unsloth Qwen3 Colab notebook. Do not mix old trainer examples with newer packages.


In [ ]:
%pip -q install -U unsloth datasets trl


## 03 | Import and set reproducible choices
**Why:** Decide the model, random seed and the maximum length before training so you know exactly what was used.

**What the code is doing:** `import` statements make functions available. `random`, NumPy and PyTorch have separate random-number sources. Setting them to 42 helps reproduce a run, although GPU software can still vary. `MAX_LENGTH=512` caps the text pieces used in one training record. `MODEL_ID` points to a real 4 bit Qwen3 0.6B model on Hugging Face. 0.6B means roughly 600 million parameters; it is an intentionally small practice model.

**Expected:** You should see the printed checkpoint name.

**Check:** In the lab report, write down model name, date, GPU type and version of the notebook.

**If it fails:** If `unsloth` fails to import after installation, restart the Colab runtime and rerun the import cell.


In [ ]:
import random
import numpy as np
import torch
from unsloth import FastLanguageModel
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
MAX_LENGTH = 512
MODEL_ID = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"
print("Chosen checkpoint:", MODEL_ID)


## 04 | Download the starting model
**Why:** Load a pre-trained model before it learns anything from your examples. This is the baseline.

**What the code is doing:** `FastLanguageModel.from_pretrained` downloads or reads the model. `model` holds the neural network; `tokenizer` turns words into numerical tokens and back again. `max_seq_length` controls the allowed training context. `load_in_4bit=True` stores base weights in an efficient format to save VRAM. This is NOT the same as training every parameter at 4 bit. `dtype=None` asks Unsloth to select a suitable calculation type for the GPU.

**Expected:** Unsloth messages appear and finally Model and tokenizer loaded.

**Check:** Leave `MODEL_ID` unchanged for the first experiment; changing it makes comparisons harder.

**If it fails:** If download fails, check connectivity and the checkpoint URL. Some software versions or Colab GPUs require changes from the official notebook.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
print("Model and tokenizer loaded")


## 05 | Create 48 training examples and 12 unseen tests
**Why:** Provide a small dataset without any external download or personal data. The training and test topics are deliberately different.

**What the code is doing:** Each tuple is a topic, a short explanation and a code example. The loop creates four different questions per training topic. This yields 12 × 4 = 48 training records and 3 × 4 = 12 unseen test records. `train_rows` teaches the model the format; `test_rows` must NOT enter training. A test set containing merely reworded versions of training questions can exaggerate results, so even the *topics* are held out here. The sample is tiny on purpose and is not enough for a reliable production model.

**Expected:** 48 training examples, 12 unseen questions, and one record printed as a Python dictionary.

**Check:** Open several examples and check that every Meaning and Example is correct. Correct small data beats large incorrect data.

**If it fails:** The JSON illustration uses double quotes, as valid JSON requires. Check the surrounding answer for correctness and simplicity.


In [ ]:
train_topics = [
    ("variable", "a named place to store a value", "count = 3"),
    ("loop", "a way to repeat instructions", "for x in items: print(x)"),
    ("function", "a reusable block of code", "def add(a,b): return a+b"),
    ("list", "an ordered collection", "items = [1, 2]"),
    ("dictionary", "a set of named values", "person = {'age': 20}"),
    ("boolean", "a true or false value", "ready = True"),
    ("class", "a blueprint for objects", "class Car: pass"),
    ("exception", "a signal that something failed", "raise ValueError()"),
    ("API", "a way for software systems to communicate", "GET /products"),
    ("database", "an organized place to keep data", "SELECT * FROM users"),
    ("cache", "storage for results we may reuse", "cache['key'] = result"),
    ("queue", "a waiting line for work", "enqueue(job)"),
]
test_topics = [
    ("recursion", "a function calling itself", "factorial(n-1)"),
    ("index", "a structure that speeds up searching", "CREATE INDEX ix ON users(name)"),
    ("JSON", "a text format for structured information", '{"ok": true}'),
]
questions = ["Explain {}.", "What is {}?", "Define {} simply.", "Teach me {}."]

def build_rows(topics):
    records = []
    for term, meaning, example in topics:
        for q in questions:
            records.append({
                "question": q.format(term),
                "answer": f"Meaning: {meaning}.\nExample: {example}",
                "term": term,
            })
    return records

train_rows = build_rows(train_topics)
test_rows = build_rows(test_topics)
print("Training examples:", len(train_rows))
print("Unseen test questions:", len(test_rows))
print(train_rows[0])


## 06 | Convert examples into the model’s chat format
**Why:** Teach the model who is speaking and which answer belongs to which question.

**What the code is doing:** A chat model recognises speaker markers and special tokens. We create a *system* instruction (the general rule), a *user* question and the desired *assistant* answer. `tokenizer.apply_chat_template` inserts Qwen-compatible boundaries. `tokenize=False` means return readable text now; training tokenizes it later. `add_generation_prompt=False` means this is a completed example, not a request for the model to start speaking. `enable_thinking=False` requests Qwen3 non-thinking formatting for this simple task. `Dataset.from_list` turns Python dictionaries into dataset rows; `.map` transforms every row.

**Expected:** You should see special message markers and the first training question plus the desired two-line answer.

**Check:** Confirm that the final answer appears after the assistant marker and that no test question is included.

**If it fails:** If `enable_thinking` is unsupported by your tokenizer version, follow the updated Qwen3 model/Unsloth chat-template guidance; use the same template for every comparison.


In [ ]:
SYSTEM = (
    "You are a software tutor. Answer using exactly two lines: "
    "Meaning: ... and Example: ... . Keep the answer short."
)

def to_training_text(record):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": record["question"]},
        {"role": "assistant", "content": record["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_rows).map(to_training_text)
print(train_dataset[0]["text"][:800])


## 07 | Ask the original model before training
**Why:** Record a baseline so you can test whether the training accomplished anything.

**What the code is doing:** The function builds messages for an unseen question. `tokenize=True` converts text into the numbers the model sees; `.to(model.device)` moves numbers to the GPU. `model.eval()` switches off training-only behaviour. `torch.inference_mode()` disables gradient calculations to save memory. `generate` produces up to 96 new tokens; `do_sample=False` uses deterministic decoding. The slice `outputs[0][inputs.shape[1]:]` removes the prompt tokens so only the answer is returned. The list `before` stores all original answers.

**Expected:** You see three printed baseline answers. Their quality may already be good, because the system instruction itself requests the two-line format.

**Check:** Do not adjust the question text after this point. Use exactly the same 12 questions and generation settings after training.

**If it fails:** If generation is slow, reduce `max_new_tokens`. If responses contain thinking markers, verify non-thinking chat formatting and parsing; do not compare mixed formats.


In [ ]:
def generate_answer(question):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt",
    ).to(model.device)
    model.eval()
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=96,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

before = [generate_answer(row["question"]) for row in test_rows]
for q, answer in zip(test_rows[:3], before[:3]):
    print("QUESTION:", q["question"])
    print("BEFORE:", answer, "\n")


## 08 | Attach a small trainable LoRA adapter
**Why:** Train only a small set of extra weights instead of retraining every number in the model.

**What the code is doing:** `get_peft_model` adds the small trainable LoRA components. `r=8` determines their size; a larger rank generally trains more values. `lora_alpha=16` controls their scaling. `lora_dropout=0` is a common Unsloth starting setting. `target_modules` says which internal layers receive adapters; leave these values alone as a beginner. `use_gradient_checkpointing="unsloth"` trades extra computation for less memory usage. `random_state` makes adapter initialisation more repeatable. The original compressed model is mostly frozen.

**Expected:** A message confirms an adapter was added; some versions print trainable parameter counts.

**Check:** We are fine tuning with LoRA and a 4 bit base, often called QLoRA, not teaching a 27B model from scratch.

**If it fails:** If a target-module error appears, copy the Qwen3 configuration from Unsloth’s maintained example notebook.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("LoRA adapter attached")


## 09 | Configure the trainer
**Why:** Tell the trainer what to read, how many updates to make, and how cautiously to learn.

**What the code is doing:** `SFTTrainer` is the supervised fine tuning manager. `SFTConfig` stores the settings. `output_dir` is where intermediate results go. `dataset_text_field="text"` chooses the column built in Cell 06. `max_length` bounds sequence length. `packing=False` makes samples easier to reason about. `per_device_train_batch_size=1` processes one example at a time on the GPU, saving memory; `gradient_accumulation_steps=4` combines four such passes before one update. `max_steps=30` limits this educational run to 30 optimisation steps. `learning_rate=2e-4` controls weight-adjustment size. `logging_steps=5` prints loss regularly. `save_steps=30` saves a checkpoint near the end. `fp16`/`bf16` chooses precision supported by the GPU. `report_to="none"` stops optional experiment trackers. `processing_class=tokenizer` is the modern TRL argument.

**Expected:** Trainer configured, possibly after Unsloth or TRL status messages.

**Check:** Write down the settings before any tuning. If you adjust three settings simultaneously you will not know what caused an outcome.

**If it fails:** TRL changes argument names over time. If you see “unexpected keyword”, use one supported version set from Unsloth’s maintained notebook, not a mix of old and new snippets.


In [ ]:
from trl import SFTTrainer, SFTConfig

settings = SFTConfig(
    output_dir="qwen_lab_checkpoints",
    dataset_text_field="text",
    max_length=MAX_LENGTH,
    packing=False,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=30,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=30,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=settings,
)
print("Trainer configured")


## 10 | Run the actual fine tuning
**Why:** This is the step that actually changes the new trainable weights using your examples.

**What the code is doing:** `trainer.train()` feeds token sequences through the model, predicts what text comes next, compares predictions with the desired training text, calculates an error called **loss**, and updates the LoRA weights to reduce that error. Thirty steps with four accumulated mini batches means approximately 120 example presentations, with repeated exposure to the 48 training records. `training_result.metrics` prints the summary. Training on tiny repetitive data can overfit very quickly.

**Expected:** Progress and loss messages appear; the run ends with a dictionary containing training metrics. No specific final loss value is guaranteed.

**Check:** A decreasing training loss indicates the model is getting better at predicting *training text*, but it does not prove it answers new questions correctly.

**If it fails:** If CUDA runs out of memory, restart and reduce MAX_LENGTH to 256, rank to 4 or reduce other memory demands. If interrupted, you may need to rerun from loading the model.


In [ ]:
training_result = trainer.train()
print(training_result.metrics)


## 11 | Repeat the same unseen questions
**Why:** Observe behavioural changes on questions and topics the trainer never saw.

**What the code is doing:** This deliberately reuses the existing `generate_answer` function and the unchanged `test_rows`. `before` was computed from the original model; `after` is computed once the LoRA weights have been updated. Keeping the prompt, tokenizer, generation settings and test questions constant makes the comparison fairer.

**Expected:** Four before-and-after pairs print. They may or may not improve: a small experiment has no guaranteed success.

**Check:** Look for hallucinated code, missing Meaning/Example labels, unnecessary extra lines and incorrect definitions.

**If it fails:** If answers are unchanged, that is still a useful result: try a better, more varied dataset, or compare without the strict system-format instruction.


In [ ]:
after = [generate_answer(row["question"]) for row in test_rows]
for row, old, new in zip(test_rows[:4], before[:4], after[:4]):
    print("QUESTION:", row["question"])
    print("BEFORE:", old)
    print("AFTER:", new, "\n")


## 12 | Score the outputs and save an evaluation CSV
**Why:** Turn a subjective impression into a simple, auditable result. Keep your raw responses for later inspection.

**What the code is doing:** `format_ok` checks whether there are exactly two nonempty lines, with the requested labels. The percentages are computed over all 12 test questions. `csv.writer` saves each question and both responses to a local report file. **Important:** this metric tests formatting only; it does not measure whether the explanations or code snippets are correct. Because the system prompt already specifies the format, the baseline score may be 100%. A score of 100% after training therefore does not prove that fine tuning helped.

**Expected:** Two scores print between 0% and 100%, and a CSV file appears in the Colab file sidebar.

**Check:** For a proper review, manually grade meaning accuracy and example correctness (0 or 1 each), and try 20+ genuinely new questions.

**If it fails:** If scores are equal, test without the exact-format system instruction and compare again; otherwise prompting and fine tuning can be confused.


In [ ]:
import csv

def format_ok(answer):
    lines = [line.strip() for line in answer.splitlines() if line.strip()]
    return (len(lines) == 2 and
            lines[0].startswith("Meaning:") and
            lines[1].startswith("Example:"))

before_score = sum(format_ok(x) for x in before) / len(before)
after_score = sum(format_ok(x) for x in after) / len(after)
print(f"Before format score: {before_score:.1%}")
print(f"After  format score: {after_score:.1%}")

with open("qwen_lab_evaluation.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["question", "before", "after", "before_format_ok", "after_format_ok"])
    for row, old, new in zip(test_rows, before, after):
        writer.writerow([row["question"], old, new, format_ok(old), format_ok(new)])
print("Saved qwen_lab_evaluation.csv")


## 13 | Save and download the LoRA adapter
**Why:** Do not lose the weights when the temporary Colab session shuts down.

**What the code is doing:** `save_pretrained` writes the trained LoRA adapter and tokenizer configuration to a folder. The `zip` shell command compresses the folder. `files.download` asks your browser to save that ZIP to your own computer. The ZIP contains the adapter, **not necessarily a standalone full model**; using it later requires the compatible original Qwen checkpoint. Colab files can disappear when a runtime resets.

**Expected:** A browser download prompt appears. Save the ZIP somewhere safe, not only in Colab.

**Check:** Unzip and confirm there is an adapter config and at least one adapter weights file.

**If it fails:** If the download popup is blocked, use the Colab Files sidebar or copy the ZIP to mounted Google Drive.


In [ ]:
model.save_pretrained("qwen_tutor_adapter")
tokenizer.save_pretrained("qwen_tutor_adapter")
!zip -qr qwen_tutor_adapter.zip qwen_tutor_adapter
from google.colab import files
files.download("qwen_tutor_adapter.zip")


## 14 | Optional: export the full result to GGUF
**Why:** Create a file format that Ollama can load as a complete inference model.

**What the code is doing:** `save_pretrained_gguf` asks Unsloth to combine/export the learned result and convert it to GGUF, a common local inference format. `q4_k_m` is a practical 4 bit output compression method; it is independent of the LoRA rank. `glob` lists the actual generated filenames. This export is **optional** because a free Colab runtime may not have enough memory or disk space for conversion, even when training succeeded.

**Expected:** If successful, the printed list contains a .gguf filename inside qwen_tutor_gguf.

**Check:** Keep the exact printed filename for the Ollama Modelfile. Do not guess it.

**If it fails:** If GGUF conversion runs out of memory, stop: the training experiment is still complete. Keep the adapter ZIP and follow Unsloth’s current export guide another day.


In [ ]:
# Run ONLY if you want to use the tuned model in Ollama.
# Export can take extra RAM, VRAM, disk space and time.
model.save_pretrained_gguf(
    "qwen_tutor_gguf", tokenizer,
    quantization_method="q4_k_m",
)
import glob
print(glob.glob("qwen_tutor_gguf/*.gguf"))


## Extra evaluation to try after the main lab
1. Write 10 questions on different computer-science topics not in the training dataset.
2. Repeat them with the exact same settings before and after.
3. Manually score factual accuracy separately from style compliance.
4. Make another evaluation run without spelling out the two-line format in the system prompt. This helps reveal what the model learned instead of just obeying the prompt.
5. Record failures rather than choosing only the best examples.

## After exporting to GGUF
Download the GGUF with Colab Files sidebar. On Windows 11, make a text file named `Modelfile` next to your downloaded GGUF with `FROM ./ACTUAL_FILENAME.gguf`. Then in PowerShell run `ollama create qwen-tutor -f Modelfile` followed by `ollama run qwen-tutor`. Ollama must already be installed.

References: https://unsloth.ai/docs/models/tutorials/qwen3-how-to-run-and-fine-tune , https://unsloth.ai/docs/get-started/unsloth-notebooks , https://research.google.com/colaboratory/faq.html , https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf
